In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "no_of_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

print(f"\nGPT_CONFIG_124M : {GPT_CONFIG_124M}")

In [ ]:
# test tiny gpt
import sys
sys.path.append("..") 
from src.tinygpt import GPTModel
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")
batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

tokens1 = tokenizer.encode(txt1)
tokens2 = tokenizer.encode(txt2)

print(f"\n\"{txt1}\"  => {tokens1}")
print(f"\n\"{txt2}\"  => {tokens2}")

batch = []

batch.append(torch.tensor(tokens1))
batch.append(torch.tensor(tokens2))

batch = torch.stack(batch, dim = 0)

print(f"\nbatch shape : {batch.shape}")
print(f"\nbatch: {batch}")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

logits = model(batch)
print(f"\nOutput shape: {logits.shape}")
print(logits)


In [ ]:
import torch
import torch.nn as nn
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
import matplotlib.pyplot as plt
gelu, relu = GELU(), nn.ReLU()

x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)
plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn

class ShortcutConnectionNeuralNet(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()

        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(layer_sizes[0][0], layer_sizes[0][1]),
                GELU()
            ),
            nn.Sequential(
                nn.Linear(layer_sizes[1][0], layer_sizes[2][1]),
                GELU()
            ),
            nn.Sequential(
                nn.Linear(layer_sizes[2][0], layer_sizes[2][1]),
                GELU()
            ),
            nn.Sequential(
                nn.Linear(layer_sizes[3][0], layer_sizes[3][1]),
                GELU()
            ),
            nn.Sequential(
                nn.Linear(layer_sizes[4][0], layer_sizes[4][1]),
                GELU()
            )
        ])

    def forward(self,x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and layer_output.shape == x.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

In [ ]:
# define a model using class ShortcutConnectionNeuralNet
torch.manual_seed(123)
layers_size = [[3,3], [3,3], [3,3],[3,3] ,[3,1]]
x = torch.tensor([[1.,0.,-1.]])
Y = torch.tensor([[0.]])
model = ShortcutConnectionNeuralNet(layers_size, False)
output = model(x)
loss_fn = nn.MSELoss()
loss = loss_fn(output, Y)

loss.backward()

for name, parameter in model.named_parameters():
    if "weight" in name:
        print(f"\n{name} has gradient mean of {parameter.grad.abs().mean().item():.11f}")

# Vanishing Gradient
We can se in the above output that as the gradient calculation is done the previous layers get smaller gradient values than current layers and the first layer gets very small gradient value. This phenomenon is known as vanishing gradient